kernel : Python (Pixi)

#### Standardizing Cluster Interpretation and Cell Type Annotation

Public single-cell datasets can be standardized, clustered, and analyzed for differential expression, yet still remain biologically ambiguous when clusters lack curated cell type labels. 
Clusters generated by unsupervised algorithms reflect mathematical similarity in gene expression space, but they do not inherently correspond to known biological cell types or functional states. 
Therefore, cell type annotation is required to assign biological identities to clusters by comparing cluster-level DE and expression patterns against established cell-type marker sets to support interpretable, reproducible downstream analyses.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

series_name = "GSE118257"
clustered_data_location = find_env_dir("CLUSTERED_DATA")
deseq_location = find_env_dir("DESEQ")

clustered_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))
de_result = pd.read_csv(os.path.join(deseq_location, series_name + "_deseq2.csv"), index_col=0)
de_result.set_index("gene", inplace=True)
de_result.sort_values("log2FoldChange_shrunk", ascending=False, inplace=True)

LOG_FOLD_CHANGE_THRESHOLD = 2
ADJUSTED_PVALUE_THRESHOLD = 0.05

/home/sungjune222/projects/protocols/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
config = {
                "color": "condition",
                "title": "Condition Distribution",
                "legend_loc": "best",
                "palette": None,
            }

plot.plot_umap(clustered_adata, series_name, has_celltype=True, 
               highlight_cells=["OPCs", "Neuron1", "Neuron2", "Neuron3", "Neuron4", "Neuron5", 
                                "Endothelial_cells1", "Endothelial_cells2", "COPs", "ImOlGs", "Endothelial_cells1", "Endothelial_cells2",
                                "Pericytes", "Astrocytes", "Astrocytes2", "Microglia_Macrophages", "Microglia_Macrophages",
                                "Oligo1", "Oligo2", "Oligo3", "Oligo4", "Oligo5", "Oligo6", "Macrophages"], 
               additional_config = [config])

In [ ]:
cell_marker_genes = {
    "Newly Formed": ["BCAS1", "ENPP6", "GALC", "TCP11L2"],
    "OPC": ["PDGFRA", "OLIG1", "OLIG2", "PCDH15", "MEGF11", "VCAN",  "MYRF", ],
    "COP": ["BCAN", "ITPR2", "GPR17", "BMP4"],
    "Oligodendrocyte": ["MBP", "PLP1", "MOG", "MAG", "CNP", "MOBP", "CLDN11", "FA2H", 
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8"],
    "Neuron": ["MAP2", "STMN2", "RBFOX3", "SNAP25", "ENO2", "SLC17A7", "SYT1", "CCK", 
               "GABRA1", "GRIN1", "SATB2", "GRIN2B","MYT1L", "SYN1", "TMEM130"],
    "NANI": ["APOE", "CD74"],
    "New Neuron": ["DCX", "GAD1", "GAD2", "CALB2", "RELN"],
    "Astrocyte": ["GFAP", "AQP4", "ALDH1L1", "SOX9", "MLC1", "ATP1B2", "GJA1", "SLC14A1", 
                  "ALDH1A1", "DIO2", "AGT", "ATP13A4", "BMPR1B", "CHI3L1", 
                  "GLIS3", "ID4", "RFX4", "SLC4A4", "SOX2",],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "TREM2", "ITGAM", "ITGAX", "SLC2A5"],
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1"],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3"],
    "Ependymal": ["FOXJ1", "DYNLRB2", "PIFO"],
    "Smooth Muscle": ["MYH11", "CNN1", "TAGLN", "ACTA2",],
    "DEG": ["HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1"],
    "Marker": ["LHFPL3", "LUZP2", "DSCAM", "XYLT1", "EPN2"],
    
    "Lymphocyte": ["CD3D", "CD3E", "CD4", "CD8A", "NKG7", "TRAC"],
    "Macrophage": ["CD68", "MRC1", "LYZ"],
    "VLMC": ["COL1A2", "COL3A1", "DCN"],
    "Epithelial": ["KRT18", "EPCAM"],
}
plot.plot_dotplot(clustered_adata, series_name, cell_marker_genes, group = "leiden")

Identify differentially expressed genes in a specific cluster

In [1]:
cluster = 1
subset = de_result[de_result["cluster"] == cluster]
subset = subset[(subset["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & (subset["padj"] < ADJUSTED_PVALUE_THRESHOLD)]
subset.head(20)

NameError: name 'de_result' is not defined

Identify clusters where a specific gene is differentially expressed

In [44]:
gene = "NKAIN2"
subset = de_result.loc[gene]
subset = subset[(subset["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & (subset["padj"] < ADJUSTED_PVALUE_THRESHOLD)]
subset.head()

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster,contrast,cluster_key
gene,,,,,,,,,,,


Identify 

In [132]:
# The ratio threshold for filtering unique markers
# The ratio is determined by (highest normalized mean count) / (second highest normalized count)
UNIQUE_THRESHOLD = 2

def filter_unique_markers(adata: AnnData, deseq_results: pd.DataFrame, cluster_key="leiden", unique_threshold=UNIQUE_THRESHOLD):
    candidates = deseq_results[
        (deseq_results["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & 
        (deseq_results["padj"] < ADJUSTED_PVALUE_THRESHOLD)
    ].copy()
    candidates.reset_index(inplace=True)
    candidates["cluster"] = candidates["cluster"].astype(str)
    needed_genes = candidates["gene"].unique()

    assert isinstance(adata.X, csr_matrix)
    library_sizes = np.asarray(adata.X.sum(axis=1)).flatten()
    library_sizes[library_sizes == 0] = 1.0
    adata_needed_genes = adata[:, needed_genes].to_memory()
 
    cluster_means_dict = dict()
    TARGET_SUM = 1e4

    for cluster in adata.obs[cluster_key].unique():
        mask = (adata.obs[cluster_key] == cluster).values
        adata_masked = adata_needed_genes[mask]
        libs = library_sizes[mask]
        scaling_factors = TARGET_SUM / libs
        
        assert isinstance(adata_masked.X, csr_matrix)
        row_scale = sparse.diags(scaling_factors)
        X_norm = row_scale @ adata_masked.X
        mean_val = np.asarray(X_norm.mean(axis=0)).flatten()

        cluster_means_dict[cluster] = mean_val
    
    cluster_means = pd.DataFrame.from_dict(cluster_means_dict, orient="index", columns=needed_genes)
    unique_markers = []

    for cluster in candidates["cluster"].unique():
        cluster_sub = candidates[candidates["cluster"] == cluster]
        genes = cluster_sub["gene"].values

        my_means = cluster_means.loc[cluster, genes].values
        rest_means: pd.DataFrame = cluster_means.drop(index=cluster)[genes].max(axis=0).values
        safe_rest_means = np.maximum(rest_means, 1e-9)
        
        ratios = my_means / safe_rest_means
        pass_mask = ratios >= unique_threshold
        
        valid_rows = cluster_sub[pass_mask].copy()
        valid_rows["specificity_ratio"] = ratios[pass_mask]
        valid_rows["my_mean"] = my_means[pass_mask]
        valid_rows["second_best_mean"] = rest_means[pass_mask]

        unique_markers.append(valid_rows)
        
    return pd.concat(unique_markers, ignore_index = True)

In [133]:
markers = filter_unique_markers(clustered_adata, de_result)
markers.head(1)

,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster,contrast,cluster_key,specificity_ratio,my_mean,second_best_mean
0,HAS2,6.733871,7.612543,1.181609,6.442525,1.175018e-10,1.177869e-08,8.112331,1.422748,1,rest,leiden,3.24211,0.346579,0.106899


In [8]:
cluster_markers = markers[(markers["cluster"] == "0") & (markers["my_mean"] > 10)]
cluster_markers.head(20)

,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster,contrast,cluster_key,specificity_ratio,my_mean,second_best_mean
15,LHFPL3,265.485458,6.359599,0.758430,8.385219,5.063120e-17,3.120263e-14,6.127264,0.872349,0,rest,leiden,25.702314,48.484253,1.886377
17,PTPRZ1,237.489503,6.086739,0.436301,13.950783,3.111681e-44,4.218818e-40,6.023811,0.442926,0,rest,leiden,5.028281,41.381935,8.229837
18,TNR,272.917776,6.051969,0.515344,11.743542,7.622585e-32,3.444900e-28,5.960382,0.538863,0,rest,leiden,18.230698,46.837742,2.569169
29,VCAN,178.979082,5.430510,0.506658,10.718306,8.351872e-27,2.830867e-23,5.330865,0.526850,0,rest,leiden,4.429077,27.972378,6.315622
32,PCDH15,177.195219,5.314781,0.646285,8.223583,1.975112e-16,1.164285e-13,5.133584,0.715501,0,rest,leiden,3.941832,30.442297,7.722880
36,SOX6,126.931750,4.965322,0.623101,7.968727,1.603169e-15,7.495093e-13,4.790322,0.677732,0,rest,leiden,3.122646,19.837198,6.352690
48,LUZP2,126.244554,4.604186,0.670926,6.862429,6.769931e-12,1.582530e-09,4.371201,0.752302,0,rest,leiden,4.892169,19.315420,3.948233
50,DSCAM,368.278707,4.379021,0.471406,9.289269,1.553517e-20,3.008940e-17,4.275496,0.494019,0,rest,leiden,5.527716,60.384514,10.923954
56,LINC00511,58.673695,4.092089,0.526640,7.770181,7.837382e-15,3.320601e-12,3.951307,0.546431,0,rest,leiden,3.865742,10.377419,2.684457
59,LRRC4C,373.963254,4.038139,0.513975,7.856684,3.944369e-15,1.782592e-12,3.901386,0.549733,0,rest,leiden,3.354597,57.578056,17.163927


In [189]:
cell_marker_genes = {
    "Nervous System": ["CACNA2D1", "NCAM1", "NFASC", "CADM2", "NRXN1", ],
    "OPC": ["PDGFRA", "OLIG1", "OLIG2", "PCDH15", "MEGF11", "VCAN",  "MYRF", ],
    "Oligodendrocyte": ["MBP", "PLP1", "MOG", "MAG", "CNP", "MOBP", "CLDN11", "FA2H", 
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8"],
    "Neuron": ["MAP2", "STMN2", "RBFOX3", "SNAP25", "ENO2", "SLC17A7", "SYT1", "CCK", 
               "GABRA1", "GRIN1", "SATB2", "GRIN2B","MYT1L", "SYN1", "TMEM130"],
    "New Neuron": ["DCX", "GAD1", "GAD2", "CALB2", "RELN"],
    "Astrocyte": ["GFAP", "AQP4", "ALDH1L1", "SOX9", "MLC1", "ATP1B2", "GJA1", "SLC14A1", 
                  "ALDH1A1", "DIO2", "AGT", "ATP13A4", "BMPR1B", "CHI3L1", 
                  "GLIS3", "ID4", "RFX4", "SLC4A4", "SOX2",],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "TREM2", "ITGAM", "ITGAX", "SLC2A5"],
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1"],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3"],
    "Ependymal": ["FOXJ1", "DYNLRB2", "PIFO"],
    "Smooth Muscle": ["MYH11", "CNN1", "TAGLN", "ACTA2",],
    "Newly Formed": ["BCAS1", "ENPP6", "GALC"],
    "DEG": ["HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1"],
    "Marker": ["LHFPL3", "LUZP2", "DSCAM", "XYLT1", "EPN2"],
    "COP": ["BCAN", "ITPR2", "GPR17", "BMP4"],
    
    "Lymphocyte": ["CD3D", "CD3E", "CD4", "CD8A", "NKG7", "TRAC"],
    "Macrophage": ["CD68", "MRC1", "LYZ"],
    "VLMC": ["COL1A2", "COL3A1", "DCN"],
    
    "Epithelial": ["KRT18", "EPCAM"],
}
plot.plot_dotplot(clustered_adata, cell_marker_genes)

In [ ]:
leiden_to_celltype = {
    "0": "OPC",
    "1": "Oligodendrocyte",
    "2": "Oligodendrocyte",
    "3": "Microglia",
    "4": "Neuron",
    "5": "Neuron",
    "6": "Neuron",
    "7": "Neuron",
    "8": "Neuron",
    "9": "Astrocyte",
    "10": "Neuron",
    "11": "VLMC",
    "12": "Endothelial",
    "13": "Neuron",
    "14": "Oligodendrocyte",
    "15": "Oligodendrocyte",
    "16": "Astrocyte",
    "17": "Oligodendrocyte",
    "18": "Oligodendrocyte",
    "19": "Endothelial",
    "20": "Astrocyte",
    "21": "Astrocyte",
    "22": "Lymphocyte",
    "23": "Macrophage",
    "24": "Macrophage",
    "25": "Lymphocyte",
    }

In [ ]:
cell_marker_genes = {
    "Musculature": [
        "Kit", "Lrig1", "Prkg1", "Pcdh7", "Foxp2", 
        "Cpe", "Meis2", "Acta2", "Tagln", "Mylk", 
        "Actg2", "Myh11"
    ],
    "Vasculature": [
        "Kdr", "Ecscr", "Cldn5", "Pecam1", "Lyve1", "Flt1"
    ],
    "ENS": [
        "Snap25", "Tubb2b", "Phox2b", "Vip", "Tac1", 
        "Chd5", "Stmn2", "S100b", "Plp1", "Gfap", 
        "Sox10", "Gpm6b", "Ptprz1"
    ],
    "Immune": [
        "Ptprc", "Ikzf1", "Ikzf3", "Arhgdib", "Coro1a", 
        "Ccr5", "Cd4", "Itgax", "Cd8a", "Cd3g", "Cd3d"
    ],
    "Mesenchymal": [
        "Vim", "Col1a1", "Cd81", "Cd34", "Col1a2", 
        "Dcn", "Wt1", "Thy1", "Pdpn", "Col3a1", 
        "Myrf", "Bmp5", "Pdgfra"
    ]
}
plot.plot_dotplot(clustered_adata, cell_marker_genes)

In [ ]:
leiden_to_celltype = {
    "0": "??",
    "1": "??",
    "2": "ENS",
    "3": "??",
    "4": "??",
    "5": "??",
    "6": "??",
    "7": "??",
    "8": "??",
    "9": "Immune",
    "10": "??",
    "11": "??",
    "12": "??",
    "13": "ENS",
    "14": "??",
    "15": "ENS",
    "16": "Vasculature",
    "17": "Vasculature",
    "18": "??",
    "19": "??",
    "20": "??",
    "21": "??",
    "22": "??",
    "23": "??",
    "24": "??",
    "25": "??",
    "26": "??",
    "27": "??",
    "28": "??",
    "29": "Vasculature",
    "30": "??",
    "31": "Immune",
    "32": "??",
    "33": "??",
    "34": "??",
    "35": "??",
    "36": "??",
    "37": "??",
    "38": "??",
    "39": "??",
    "40": "??",
    }